In [ ]:
import pandas as pd


In [ ]:

# 1. Configuración de los nombres de tus archivos CSV
archivo_papas = "dataset_unificado_volumen_precios_total.csv"              # Tu archivo con volumen, variedad, etc.
archivo_diesel = "indicadores_diesel_limpio.csv"       # El archivo que limpiamos antes
archivo_resultado = "dataset_papas_con_diesel_hasta_abril_2026.csv" # El nuevo archivo combinado

try:
    # 2. Leer los archivos CSV
    # Usamos encoding='latin1' por si el archivo de papas tiene tildes (región, variedad)
    df_papas = pd.read_csv(archivo_papas, encoding='utf-8')
    df_diesel = pd.read_csv(archivo_diesel, encoding='utf-8')

    # Estandarizar nombres de columnas a minúsculas para evitar errores de mayúsculas/minúsculas
    df_papas.columns = df_papas.columns.str.strip().str.lower()
    df_diesel.columns = df_diesel.columns.str.strip().str.lower()

    print("✔ Archivos cargados correctamente.")

    # 3. Convertir las columnas de fecha al formato datetime oficial de Pandas
    df_papas['fecha'] = pd.to_datetime(df_papas['fecha'], errors='coerce')
    df_diesel['fecha'] = pd.to_datetime(df_diesel['fecha'], errors='coerce')

    # 4. Crear la "llave" de unión usando el Año y Mes (Formato YYYY-MM)
    df_papas['anio_mes'] = df_papas['fecha'].dt.to_period('M')
    df_diesel['anio_mes'] = df_diesel['fecha'].dt.to_period('M')

    # 5. Seleccionar solo las columnas necesarias del dataset de diésel para el merge
    # Nos quedamos con 'anio_mes' (para unir) e 'indice_precio_diesel' (el valor que queremos agregar)
    df_diesel_reducido = df_diesel[['anio_mes', 'indice_precio_diesel']]

    # Si hay meses duplicados en el archivo de diésel, dejamos solo el primero para evitar que se dupliquen filas
    df_diesel_reducido = df_diesel_reducido.drop_duplicates(subset=['anio_mes'])

    # 6. Realizar el MERGE (Left join: mantiene todas las filas de las papas y añade el diésel donde coincida)
    df_final = pd.merge(df_papas, df_diesel_reducido, on='anio_mes', how='left')

    # 7. Limpieza final: eliminamos la columna temporal 'anio_mes'
    df_final = df_final.drop(columns=['anio_mes'])


    df_final = df_final[
        ~(
            (df_final["fecha"].dt.year == 2026) &
            (df_final["fecha"].dt.month == 5)
        )
    ]

    # ==========================================
    # RESET INDEX
    # ==========================================

    df_final = df_final.reset_index(drop=True)

    # 8. Guardar el nuevo dataset combinado
    df_final.to_csv(archivo_resultado, index=False, encoding='utf-8')

    print(f"✔ ¡Merge completado con éxito!")
    print(f"✔ El nuevo archivo se ha guardado en: '{archivo_resultado}'")
    print("\nMuestra del resultado final (primeros 3 registros):")
    print(df_final.head(3))

except FileNotFoundError as e:
    print(f"❌ Error: No se encontró uno de los archivos. Verifique los nombres. Detalle: {e}")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado durante el merge: {e}")

✔ Archivos cargados correctamente.
✔ ¡Merge completado con éxito!
✔ El nuevo archivo se ha guardado en: 'dataset_papas_con_diesel_hasta_abril_2026.csv'

Muestra del resultado final (primeros 3 registros):
       fecha  volumen     variedad      zona  volumen_log    region  \
0 2026-04-30     11.0  Papa Yungay     Jauja     2.484907     Junin   
1 2026-04-30     12.0  Papa Yungay  Huancayo     2.564949     Junin   
2 2026-04-30     26.0  Papa Yungay  Huamanga     3.295837  Ayacucho   

   precio_promedio_kg  indice_precio_diesel  
0                1.53              131.1866  
1                1.53              131.1866  
2                1.53              131.1866  
